# 03 — Pré-processamento

Este notebook realiza o pré-processamento dos dados para os modelos de ML.

**Etapas:**
1. Construir a matriz X e o vetor y
2. Identificar N (amostras) e p (parâmetros)
3. Separar treino e teste (com justificativa)
4. Aplicar normalização/padronização (com justificativa)

**IMPORTANTE:** O conjunto de teste não deve ser tocado até a fase final de métricas.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import joblib

from src.utils import load_processed_data, DATA_PROCESSED, MODELS_DIR
from src.preprocessing import (
    split_features_target,
    split_train_test,
    normalize_features,
    handle_missing_values,
    get_dataset_info,
    FEATURES,
)

MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
df = load_processed_data()
print(f"Dataset carregado: {df.shape[0]} municípios × {df.shape[1]} colunas")

Dataset carregado: 5570 municípios × 18 colunas


## 3.1 Tratamento de valores ausentes

O notebook 01 já imputou todos os nulos pela mediana da respectiva feature (máximo de 0,59% de nulos por coluna antes da imputação). Aqui apenas verificamos o contrato: o dataset deve ter zero nulos nas colunas de features e alvo.

In [3]:
df = handle_missing_values(df)
print("Verificação de nulos: OK — zero nulos nas colunas de features e alvo.")

Verificação de nulos: OK — zero nulos nas colunas de features e alvo.


## 3.2 Construir Matriz X e Vetor y

Separar as features socioeconômicas (X) da variável alvo (y).

**Lembrete:** Dados de crime NÃO entram em X.

In [4]:
X, y = split_features_target(df)
info = get_dataset_info(X, y)

print(f"N (amostras) = {info['N']}")
print(f"p (features) = {info['p']}")
print(f"\nFeatures ({info['p']}):")
for i, feat in enumerate(info['feature_names'], 1):
    print(f"  {i:2d}. {feat}")
print(f"\nBalanceamento da variável alvo:")
for cls, pct in info['class_balance_pct'].items():
    print(f"  Classe {cls}: {info['class_balance'][cls]} amostras ({pct*100:.1f}%)")

print(f"\nColunas EXCLUÍDAS de X (dados de crime e identificadores):")
excluidas = [c for c in df.columns if c not in FEATURES]
print(f"  {excluidas}")

N (amostras) = 5570
p (features) = 15

Features (15):
   1. pop_total
   2. pop_urbana_pct
   3. perc_jovens_15_29
   4. renda_per_capita
   5. taxa_desemprego
   6. taxa_analfabetismo_15
   7. perc_esgoto_adequado
   8. gini
   9. idhm
  10. idhm_renda
  11. idhm_longevidade
  12. idhm_educacao
  13. perc_pobres
  14. pib_per_capita
  15. densidade_demografica

Balanceamento da variável alvo:
  Classe 0: 2785 amostras (50.0%)
  Classe 1: 2785 amostras (50.0%)

Colunas EXCLUÍDAS de X (dados de crime e identificadores):
  ['cod_ibge', 'taxa_homicidios', 'alta_violencia']


## 3.3 Separação Treino/Teste

**Justificativa da proporção 80/20:**

Com N = 5.570 municípios, a divisão 80/20 resulta em:
- **Treino:** ~4.456 amostras — suficiente para treinar 3 modelos com cross-validation interna
- **Teste:** ~1.114 amostras — suficiente para obter estimativas confiáveis de E_out (margem de erro ~3% com IC 95%)

A divisão é feita com **estratificação** pela variável alvo (`stratify=y`) para garantir que o balanceamento 50%/50% seja mantido em ambos os conjuntos.

**Regra:** O conjunto de teste só será usado na fase final de comparação dos modelos (Notebook 07). Todos os hiperparâmetros serão selecionados por cross-validation sobre o treino.

In [5]:
X_train, X_test, y_train, y_test = split_train_test(X, y, test_size=0.2, random_state=42)

print(f"Treino: {len(X_train)} amostras ({len(X_train)/len(X)*100:.1f}%)")
print(f"Teste:  {len(X_test)} amostras ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nBalanceamento no treino: {y_train.value_counts(normalize=True).to_dict()}")
print(f"Balanceamento no teste:  {y_test.value_counts(normalize=True).to_dict()}")

Treino: 4456 amostras (80.0%)
Teste:  1114 amostras (20.0%)

Balanceamento no treino: {0: 0.5, 1: 0.5}
Balanceamento no teste:  {1: 0.5, 0: 0.5}


## 3.4 Normalização/Padronização

**Método escolhido: StandardScaler (padronização z-score)**

$$x' = \frac{x - \mu}{\sigma}$$

**Justificativa:**

A EDA (notebook 02) revelou que as features possuem escalas muito distintas:
- `pop_total` varia de ~833 a ~11,4 milhões
- `idhm` varia de 0,42 a 0,86
- `pib_per_capita` varia de ~5.700 a ~919.000

Sem normalização, modelos baseados em distância ou gradiente dariam peso desproporcional às features de maior magnitude.

O **StandardScaler** (média 0, desvio padrão 1) é obrigatório para:
- **SVM com kernel RBF** — distâncias no espaço de features dependem da escala
- **Rede Neural** — gradientes e a inicialização dos pesos pressupõem entradas centradas

Para a **Árvore de Decisão**, a escala não afeta os splits, mas aplicamos o mesmo pré-processamento para uniformidade experimental.

**Data leakage:** O scaler é ajustado (`fit`) **apenas nos dados de treino** e aplicado (`transform`) em ambos os conjuntos, evitando vazamento de informação do teste.

In [6]:
X_train_scaled, X_test_scaled, scaler = normalize_features(X_train, X_test, method='standard')

print("Verificação pós-normalização (treino):")
print(f"  Média por feature (deve ser ~0): {X_train_scaled.mean(axis=0).round(6)}")
print(f"  Desvio padrão por feature (deve ser ~1): {X_train_scaled.std(axis=0).round(6)}")
print(f"\nVerificação pós-normalização (teste) — ligeiramente diferente de 0/1, esperado:")
print(f"  Média: {X_test_scaled.mean(axis=0).round(3)}")
print(f"  Std:   {X_test_scaled.std(axis=0).round(3)}")
print(f"\nShape treino: {X_train_scaled.shape}")
print(f"Shape teste:  {X_test_scaled.shape}")

Verificação pós-normalização (treino):
  Média por feature (deve ser ~0): [ 0.  0.  0.  0. -0. -0. -0. -0.  0.  0.  0. -0. -0. -0.  0.]
  Desvio padrão por feature (deve ser ~1): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]

Verificação pós-normalização (teste) — ligeiramente diferente de 0/1, esperado:
  Média: [ 0.005  0.071 -0.082  0.011 -0.031 -0.018  0.056  0.002  0.043  0.047
  0.043  0.033 -0.055  0.018 -0.002]
  Std:   [0.616 0.992 0.951 0.937 0.983 0.992 1.009 0.974 0.972 0.983 0.973 0.975
 0.978 0.947 0.909]

Shape treino: (4456, 15)
Shape teste:  (1114, 15)


## 3.5 Salvar dados pré-processados

Salvar os arrays de treino/teste para uso nos notebooks de modelagem.

In [7]:
import numpy as np

# Salvar arrays numpy
np.save(DATA_PROCESSED / 'X_train.npy', X_train_scaled)
np.save(DATA_PROCESSED / 'X_test.npy', X_test_scaled)
np.save(DATA_PROCESSED / 'y_train.npy', y_train.values)
np.save(DATA_PROCESSED / 'y_test.npy', y_test.values)

# Salvar scaler para uso nos notebooks de modelos
joblib.dump(scaler, MODELS_DIR / 'scaler.joblib')

# Salvar nomes das features para referência
feature_names_path = DATA_PROCESSED / 'feature_names.txt'
feature_names_path.write_text('\n'.join(FEATURES))

print("Arquivos salvos:")
print(f"  {DATA_PROCESSED / 'X_train.npy'}  — shape {X_train_scaled.shape}")
print(f"  {DATA_PROCESSED / 'X_test.npy'}   — shape {X_test_scaled.shape}")
print(f"  {DATA_PROCESSED / 'y_train.npy'}  — shape {y_train.shape}")
print(f"  {DATA_PROCESSED / 'y_test.npy'}   — shape {y_test.shape}")
print(f"  {MODELS_DIR / 'scaler.joblib'}")
print(f"  {feature_names_path}")
print("\nPré-processamento concluído. Conjunto de teste isolado até o Notebook 07.")

Arquivos salvos:
  /home/gabriel/Projetos/socio_violence_predictor/notebooks/../data/processed/X_train.npy  — shape (4456, 15)
  /home/gabriel/Projetos/socio_violence_predictor/notebooks/../data/processed/X_test.npy   — shape (1114, 15)
  /home/gabriel/Projetos/socio_violence_predictor/notebooks/../data/processed/y_train.npy  — shape (4456,)
  /home/gabriel/Projetos/socio_violence_predictor/notebooks/../data/processed/y_test.npy   — shape (1114,)
  /home/gabriel/Projetos/socio_violence_predictor/notebooks/../models/scaler.joblib
  /home/gabriel/Projetos/socio_violence_predictor/notebooks/../data/processed/feature_names.txt

Pré-processamento concluído. Conjunto de teste isolado até o Notebook 07.
